# W4: F&B 매출 & 주문 분석

이 노트북은 F&B 매장의 매출 데이터를 분석하고 시각화하는 도구입니다.

## Step 0: 라이브러리 설치 및 폰트 설정

In [ ]:
# 필요한 라이브러리 설치
!pip install pandas matplotlib -q

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import numpy as np
from IPython.display import display, Image
import io
import base64
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정
def setup_korean_font():
    try:
        # 시스템에 설치된 폰트 찾기
        font_files = fm.findSystemFonts()
        korean_fonts = []
        
        # 한국어 폰트 찾기
        for font_file in font_files:
            try:
                font_prop = fm.FontProperties(fname=font_file)
                font_name = font_prop.get_name()
                # 한국어 폰트인지 확인 (일반적인 한국어 폰트 이름 포함)
                if any(keyword in font_name.lower() for keyword in ['malgun', 'gulim', 'dotum', 'batang', 'nanum', 'apple sd gothic', 'noto sans']):
                    korean_fonts.append(font_file)
                    print(f"한국어 폰트 찾음: {font_name} - {font_file}")
            except:
                continue
        
        if korean_fonts:
            # 찾은 첫 번째 한국어 폰트 사용
            selected_font = korean_fonts[0]
            fm.fontManager.addfont(selected_font)
            plt.rc('font', family=fm.FontProperties(fname=selected_font).get_name())
            print(f"폰트 설정 완료: {fm.FontProperties(fname=selected_font).get_name()}")
        else:
            # 한국어 폰트가 없는 경우 기본 폰트 사용
            print("경고: 시스템에서 한국어 폰트를 찾지 못했습니다. 영문으로 표시됩니다.")
            plt.rc('font', family='DejaVu Sans')
    except Exception as e:
        print(f"폰트 설정 중 오류 발생: {e}")
        plt.rc('font', family='DejaVu Sans')

# 마이너스 부호 표시 설정
plt.rcParams['axes.unicode_minus'] = False

# 폰트 설정 실행
setup_korean_font()

## Step 1: 가게 정보 입력

In [ ]:
# 가게 정보 입력
store_name = input("가게 이름을 입력하세요: ")
platform = input("플랫폼을 입력하세요 (예: 배달의민족, 요기요, 쿠팡이츠): ")

print(f"\n가게 정보:\n- 이름: {store_name}\n- 플랫폼: {platform}")

## Step 2: 데이터 로드

In [ ]:
# GitHub에서 샘플 데이터 로드
try:
    url = "https://raw.githubusercontent.com/Reasonofmoon/hexa-2/main/shared/fnb_sales_sample.csv"
    df = pd.read_csv(url)
    print("✅ GitHub에서 데이터 로드 완료")
    print(f"데이터形状: {df.shape}")
    display(df.head())
except Exception as e:
    print(f"❌ GitHub 데이터 로드 실패: {e}")
    print("\n파일 업로드를 시도하시겠습니까? (files.upload() 사용)")
    user_choice = input("y/n: ").lower()
    
    if user_choice == 'y':
        try:
            from google.colab import files
            uploaded = files.upload()
            if uploaded:
                filename = list(uploaded.keys())[0]
                df = pd.read_csv(io.StringIO(uploaded[filename].decode('utf-8')))
                print(f"✅ 파일 업로드 완료: {filename}")
                display(df.head())
            else:
                print("❌ 파일 업로드가 취소되었습니다.")
                print("샘플 데이터를 생성합니다...")
                df = create_sample_data()
        except Exception as upload_error:
            print(f"❌ 파일 업로드 실패: {upload_error}")
            print("샘플 데이터를 생성합니다...")
            df = create_sample_data()
    else:
        print("샘플 데이터를 생성합니다...")
        df = create_sample_data()

def create_sample_data():
    """샘플 F&B 판매 데이터 생성"""
    import random
    from datetime import datetime, timedelta
    
    menus = ['김치찌개', '된장찌개', '순두부찌개', '부대찌개', '비빔밥', '불고기', '갈비', '냉면', '떡볶이', '라면']
    prices = [8000, 7000, 9000, 12000, 10000, 13000, 18000, 9000, 6000, 4000]
    
    data = []
    for i in range(100):
        menu_idx = random.randint(0, len(menus) - 1)
        data.append({
            'order_id': f'ORD_{i+1:04d}',
            'menu': menus[menu_idx],
            'price': prices[menu_idx],
            'quantity': random.randint(1, 3),
            'rating': random.uniform(3.5, 5.0),
            'order_time': datetime.now() - timedelta(hours=random.randint(1, 168))
        })
    
    sample_df = pd.DataFrame(data)
    print("✅ 샘플 데이터 생성 완료")
    display(sample_df.head())
    return sample_df

## Step 3: 메뉴별 매출 집계

In [ ]:
# 메뉴별 매출 집계
df['total_sales'] = df['price'] * df['quantity']

# 메뉴별 집계
menu_stats = df.groupby('menu').agg({
    'total_sales': 'sum',
    'order_id': 'count',
    'rating': 'mean'
}).rename(columns={
    'order_id': 'order_count',
    'rating': 'avg_rating'
}).round(2)

# 평균 단가 계산
menu_stats['avg_price'] = (menu_stats['total_sales'] / menu_stats['order_count']).round(0)

# 컬럼명 한글화
menu_stats.columns = ['총매출액', '주문수', '평균평점', '평균단가']

# 정렬
menu_stats = menu_stats.sort_values('총매출액', ascending=False)

print(f"\n📊 {store_name} - 메뉴별 매출 분석")
display(menu_stats)

## Step 4: 데이터 시각화

In [ ]:
# 시각화 설정
plt.figure(figsize=(15, 10))

# 1. 메뉴별 매출 막대그래프
plt.subplot(2, 2, 1)
colors = plt.cm.Set3(np.linspace(0, 1, len(menu_stats)))
bars = plt.bar(menu_stats.index, menu_stats['총매출액'], color=colors)
plt.title('메뉴별 총매출액', fontsize=14, fontweight='bold')
plt.xlabel('메뉴')
plt.ylabel('매출액 (원)')
plt.xticks(rotation=45)

# 막대 위에 숫자 표시
for bar, value in zip(bars, menu_stats['총매출액']):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(menu_stats['총매출액'])*0.01, 
             f'{value:,.0f}', ha='center', va='bottom', fontsize=9)

# 2. 주문 비율 파이차트
plt.subplot(2, 2, 2)
plt.pie(menu_stats['주문수'], labels=menu_stats.index, autopct='%1.1f%%', 
        colors=colors, startangle=90)
plt.title('메뉴별 주문 비율', fontsize=14, fontweight='bold')

# 3. 평균평점 막대그래프
plt.subplot(2, 2, 3)
rating_bars = plt.bar(menu_stats.index, menu_stats['평균평점'], color='skyblue')
plt.title('메뉴별 평균평점', fontsize=14, fontweight='bold')
plt.xlabel('메뉴')
plt.ylabel('평점')
plt.xticks(rotation=45)
plt.ylim(3.0, 5.0)

# 막대 위에 평점 표시
for bar, value in zip(rating_bars, menu_stats['평균평점']):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05, 
             f'{value:.1f}', ha='center', va='bottom', fontsize=9)

# 4. 주문수 vs 매출액 산점도
plt.subplot(2, 2, 4)
plt.scatter(menu_stats['주문수'], menu_stats['총매출액'], 
           s=menu_stats['평균단가']/10, alpha=0.6, c=colors, edgecolors='black')
plt.title('주문수 vs 매출액', fontsize=14, fontweight='bold')
plt.xlabel('주문수')
plt.ylabel('매출액 (원)')

# 메뉴 이름 레이블
for i, menu in enumerate(menu_stats.index):
    plt.annotate(menu, 
                 (menu_stats['주문수'].iloc[i], menu_stats['총매출액'].iloc[i]),
                 xytext=(5, 5), textcoords='offset points', fontsize=8)

plt.tight_layout()
plt.show()

# 전체 요약 정보
total_revenue = menu_stats['총매출액'].sum()
total_orders = menu_stats['주문수'].sum()
avg_rating_overall = menu_stats['평균평점'].mean()
best_menu = menu_stats.index[0]

print(f"\n🎯 {store_name} 전체 요약")
print(f"총매출액: {total_revenue:,.0f}원")
print(f"총주문수: {total_orders:,}건")
print(f"전체 평균평점: {avg_rating_overall:.2f}점")
print(f"최고 매출 메뉴: {best_menu} ({menu_stats['총매출액'].iloc[0]:,.0f}원)")

## Step 5: 데이터 내보내기

In [ ]:
# CSV 파일로 저장
import os
from datetime import datetime

# 파일 이름 생성
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
csv_filename = f"{store_name}_매출분석_{timestamp}.csv"
chart_filename = f"{store_name}_매출차트_{timestamp}.png"

# CSV 저장
menu_stats.to_csv(csv_filename, encoding='utf-8-sig')
print(f"✅ CSV 파일 저장 완료: {csv_filename}")

# 차트 PNG 저장
plt.figure(figsize=(12, 8))

# 메인 차트 생성
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))

# 1. 총매출액 막대그래프
bars1 = ax1.bar(menu_stats.index, menu_stats['총매출액'], color=colors)
ax1.set_title('메뉴별 총매출액', fontsize=14, fontweight='bold')
ax1.set_xlabel('메뉴')
ax1.set_ylabel('매출액 (원)')
ax1.tick_params(axis='x', rotation=45)

for bar, value in zip(bars1, menu_stats['총매출액']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(menu_stats['총매출액'])*0.01, 
             f'{value:,.0f}', ha='center', va='bottom', fontsize=9)

# 2. 주문 비율 파이차트
ax2.pie(menu_stats['주문수'], labels=menu_stats.index, autopct='%1.1f%%', 
        colors=colors, startangle=90)
ax2.set_title('메뉴별 주문 비율', fontsize=14, fontweight='bold')

# 3. 평균평점
bars3 = ax3.bar(menu_stats.index, menu_stats['평균평점'], color='skyblue')
ax3.set_title('메뉴별 평균평점', fontsize=14, fontweight='bold')
ax3.set_xlabel('메뉴')
ax3.set_ylabel('평점')
ax3.tick_params(axis='x', rotation=45)
ax3.set_ylim(3.0, 5.0)

for bar, value in zip(bars3, menu_stats['평균평점']):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05, 
             f'{value:.1f}', ha='center', va='bottom', fontsize=9)

# 4. 요약 정보 텍스트
ax4.axis('off')
summary_text = f"""
{store_name} 매출 분석 요약
==========================

총매출액: {total_revenue:,.0f}원
총주문수: {total_orders:,}건
전체 평균평점: {avg_rating_overall:.2f}점
최고 매출 메뉴: {best_menu}
플랫폼: {platform}

TOP 3 메뉴:
"""

for i, (menu, row) in enumerate(menu_stats.head(3).iterrows(), 1):
    summary_text += f"\n{i}. {menu}: {row['총매출액']:,.0f}원 ({row['주문수']}건)"

ax4.text(0.1, 0.9, summary_text, transform=ax4.transAxes, fontsize=12,
         verticalalignment='top', fontfamily='monospace')

plt.tight_layout()
plt.savefig(chart_filename, dpi=300, bbox_inches='tight')
plt.show()

print(f"✅ 차트 이미지 저장 완료: {chart_filename}")

In [ ]:
# 다운로드 기능 (Colab 환경에서만 작동)
try:
    from google.colab import files
    print("\n📥 파일 다운로드")
    files.download(csv_filename)
    files.download(chart_filename)
except ImportError:
    print(f"\n💾 파일이 로컬에 저장되었습니다:")
    print(f"- CSV: {csv_filename}")
    print(f"- 차트: {chart_filename}")
    
# 최종 확인
print(f"\n🎉 {store_name} F&B 매출 분석 완료!")
print(f"생성된 파일: {len([csv_filename, chart_filename])}개")